# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**: 런타임 유형 → GPU (T4 이상)

**순서**: [1] → [2] → [3] → [4] → [5] → [6] → [7] → [8]

이 노트북은 `Run All`로 `RUN_RANK` 하나만 실행한다.
`RUN_RANK=16`으로 한 번 돌리고 GitHub에 결과를 반영한 뒤, `RUN_RANK=32`로 다시 실행한다.
T4 메모리가 빡빡하면 `BATCH_SIZE=4`를 쓰고, 여유가 있으면 더 올릴 수 있다.

In [1]:
# [1] GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU 없음: 런타임 > 런타임 유형 변경 > T4 GPU'
print(f'torch  {torch.__version__}')
print(f'GPU    {torch.cuda.get_device_name(0)}')

torch  2.11.0+cu128
GPU    NVIDIA A100-SXM4-80GB


In [2]:
# [2] 환경 세팅
# Colab 환경: torch 2.11.0+cu130 / torchvision 0.26.0+cu128 / torchaudio cu128
# torchvision/torchaudio 모두 cu128 빌드 → cu130 torch와 CUDA major 불일치
import subprocess, os, re, importlib.util

# ── 1. torchvision CUDA 버전 체크 비활성화 ───────────────────────────────────
_ext = '/usr/local/lib/python3.12/dist-packages/torchvision/extension.py'
with open(_ext) as f: _s = f.read()
_s = _s.replace('def # _check_cuda_version()  # patched:', 'def _check_cuda_version():')
_s = re.sub(r'if t_major != tv_major:', 'if False:  # patched', _s)
_s = re.sub(r'^_check_cuda_version\(\)\s*$', '# _check_cuda_version()  # patched', _s, flags=re.MULTILINE)
with open(_ext, 'w') as f: f.write(_s)
print('torchvision patched')

# ── 2. torchaudio CUDA 버전 체크 비활성화 ────────────────────────────────────
_ta = '/usr/local/lib/python3.12/dist-packages/torchaudio/_extension/utils.py'
if os.path.exists(_ta):
    with open(_ta) as f: _s = f.read()
    _s = re.sub(r'if ta_version != t_version:', 'if False:  # patched', _s)
    with open(_ta, 'w') as f: f.write(_s)
    print('torchaudio patched')

# ── 3. peft / diffusers / accelerate 설치 (--no-deps: torch 버전 유지) ───────
subprocess.run(['pip', 'install', '-q', '--no-deps',
                'peft', 'diffusers', 'accelerate'], check=True)

# ── 4. peft 패치 — torchao 구버전(0.10.0)에서 raise 대신 return False ────────
_spec = importlib.util.find_spec('peft')
if _spec:
    _pu = os.path.join(os.path.dirname(_spec.origin), 'import_utils.py')
    with open(_pu) as f: _s = f.read()
    _p = _s.replace(
        'if torchao_version < TORCHAO_MINIMUM_VERSION:',
        'if torchao_version < TORCHAO_MINIMUM_VERSION:\n        return False  # patched'
    )
    if _p != _s:
        with open(_pu, 'w') as f: f.write(_p)
        print('peft patched OK')
    else:
        print('peft: already patched or pattern not found')

# ── 5. transformers + 의존성 설치 ─────────────────────────────────────────────
subprocess.run(['pip', 'install', '-q', '-U',
                'transformers', 'tokenizers', 'huggingface-hub', 'safetensors',
                'packaging', 'h5py', 'regex', 'tqdm', 'filelock', 'requests'], check=True)

# ── 6. 검증 (fresh subprocess) ───────────────────────────────────────────────
r = subprocess.run(['python', '-c', '''
import torch, torchvision, torchaudio
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
import transformers
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"torchaudio   {torchaudio.__version__}")
print(f"diffusers    OK")
print(f"peft         OK")
print(f"transformers {transformers.__version__}")
print("ALL OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('=== ERROR ===')
    print(r.stderr[-3000:])
    raise RuntimeError('환경 검증 실패 — 위 에러 확인 필요')

torchvision patched
torchaudio patched
peft patched OK
torch        2.11.0+cu128
torchvision  0.26.0+cu128
torchaudio   2.11.0+cu128
diffusers    OK
peft         OK
transformers 5.12.1
ALL OK



In [3]:
# [3] 코드 clone / pull (S24 .npz 포함)
import os, subprocess

REPO    = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

def _clone():
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

if os.path.exists(WORKDIR):
    if not os.path.exists(os.path.join(WORKDIR, '.git')):
        subprocess.run(['rm', '-rf', WORKDIR], check=True)
        _clone()
    else:
        r = subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'])
        if r.returncode != 0:
            subprocess.run(['rm', '-rf', WORKDIR], check=True)
            _clone()
else:
    _clone()

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-3'])

npz = [f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]
ckpt = 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'
print(f'S24 .npz: {len(npz)}개,  SupCon ckpt: {os.path.exists(ckpt)}')

S24 .npz: 10개,  SupCon ckpt: True


In [4]:
# [4] Google Drive 마운트 (체크포인트 저장)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DRIVE = '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen'
os.makedirs(CKPT_DRIVE, exist_ok=True)

dst = '/content/vsvi_project/checkpoints_vsre_lora_gen'
if not os.path.exists(dst):
    os.symlink(CKPT_DRIVE, dst)
print(f'checkpoints_vsre_lora_gen -> {CKPT_DRIVE}')

Mounted at /content/drive
checkpoints_vsre_lora_gen -> /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen


In [5]:
# [5] 실행 설정 및 최종 확인
import os, torch

RUN_RANK = os.environ.get('RUN_RANK', '16')  # '16' or '32'
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '4'))
EPOCHS = int(os.environ.get('EPOCHS', '100'))
N_EEG_TOKENS = int(os.environ.get('N_EEG_TOKENS', '16'))
SUPCON_CKPT = 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon'
IMG_ROOT = 'preproc_data_vi/images'
CKPT_ROOT = 'checkpoints_vsre_lora_gen'

assert RUN_RANK in {'16', '32'}, 'RUN_RANK must be 16 or 32'
W = WORKDIR
checks = {
    'CUDA': torch.cuda.is_available(),
    'npz 10개': len([f for f in os.listdir(f'{W}/preproc_vs_re') if f.startswith('preproc_subj_24_') and f.endswith('.npz')]) == 10,
    'supcon ckpt': os.path.exists(f'{W}/checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'),
    'images 01~09': all(os.path.exists(f'{W}/preproc_data_vi/images/{i:02d}.png') for i in range(1, 10)),
}
for k, v in checks.items():
    print(f'{k:14s}: {"OK" if v else "FAIL"}')
if not all(checks.values()):
    raise RuntimeError('사전 확인 실패 -- 위 출력 확인 후 문제 해결')

print(f'RUN_RANK={RUN_RANK}')
print(f'BATCH_SIZE={BATCH_SIZE}')
print(f'EPOCHS={EPOCHS}')
print(f'N_EEG_TOKENS={N_EEG_TOKENS}')

CUDA          : OK
npz 10개       : OK
supcon ckpt   : OK
images 01~09  : OK
RUN_RANK=16
BATCH_SIZE=4
EPOCHS=100
N_EEG_TOKENS=16


In [ ]:
# [6] S24 LoRA 학습 - background + Drive log + fp16
import os, subprocess, shlex, datetime

os.chdir(WORKDIR)

LOG_DIR = "/content/drive/MyDrive/vsvi_data/logs"
os.makedirs(LOG_DIR, exist_ok=True)

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"{LOG_DIR}/s24_lora_r{RUN_RANK}_tok{N_EEG_TOKENS}_{tag}.log"
pid_path = f"{LOG_DIR}/s24_lora_r{RUN_RANK}_latest.pid"

cmd = [
    "python", "-u", "train_vs_re_lora_gen.py",
    "--subject_ids", "24",
    "--lora_r", RUN_RANK,
    "--n_eeg_tokens", str(N_EEG_TOKENS),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--img_root", IMG_ROOT,
    "--supcon_ckpt", SUPCON_CKPT,
    "--ckpt_root", CKPT_ROOT,
    "--fp16",
]

# T4/L4에서 메모리 부족하면 켜기
if torch.cuda.get_device_name(0).lower().find("t4") >= 0:
    cmd.append("--grad_ckpt")

print("Command:", shlex.join(cmd))
print("Log:", log_path)

with open(log_path, "a") as log:
    p = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

with open(pid_path, "w") as f:
    f.write(str(p.pid))

print("Started background training")
print("PID:", p.pid)
print("PID file:", pid_path)

Command: python -u train_vs_re_lora_gen.py --subject_ids 24 --lora_r 16 --n_eeg_tokens 16 --epochs 100 --batch_size 4 --img_root preproc_data_vi/images --supcon_ckpt checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon --ckpt_root checkpoints_vsre_lora_gen


In [ ]:
# [7] 결과 확인
import glob
csvs = glob.glob(f'{WORKDIR}/checkpoints_vsre_lora_gen/**/results_lora_gen.csv', recursive=True)
for p in sorted(csvs):
    print(p)
    with open(p) as f:
        print(f.read())

In [ ]:
# [8] 결과 -> PROGRESS.md 업데이트 + GitHub push
# RUN_RANK 하나만 반영한 뒤 push한다. 다음 rank는 이 셀 실행 후 다시 Run All로 별도 수행.
import os, csv, datetime, subprocess

os.chdir(WORKDIR)
subprocess.run(['git', 'pull', 'origin', 'main'], check=True)

search_roots = [
    f'{WORKDIR}/checkpoints_vsre_lora_gen',
    '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen',
]

csvs = []
for root in search_roots:
    if not os.path.exists(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        for fn in filenames:
            if fn == 'results_lora_gen.csv':
                csv_path = os.path.join(dirpath, fn)
                if csv_path not in csvs:
                    csvs.append(csv_path)
csvs.sort()

print(f'Found CSV files: {len(csvs)}')
for csv_path in csvs:
    print(' ', csv_path)

if not csvs:
    raise SystemExit('No results_lora_gen.csv found. Run training first.')

results = {}
for csv_path in csvs:
    dirname = os.path.basename(os.path.dirname(csv_path))
    r_val = None
    for part in dirname.split('_'):
        if part.startswith('r') and part[1:].isdigit():
            r_val = int(part[1:])
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            sid = int(row['sid'])
            key = (sid, r_val)
            if key not in results:
                results[key] = {**row, 'lora_r': r_val, 'dirname': dirname}

rank_key = (24, int(RUN_RANK))
if rank_key not in results:
    raise SystemExit(f'RUN_RANK={RUN_RANK} result not found.')

row = results[rank_key]
top1 = float(row['top1'])
entropy = float(row['entropy'])
dominant = float(row['dominant'])
today = datetime.datetime.now().strftime('%Y-%m-%d')

progress_path = 'PROGRESS.md'
with open(progress_path, 'a') as f:
    f.write('\n')
    f.write(f'## Track A Colab Update ({today})\n')
    f.write(f'- S24 r={RUN_RANK} Colab run completed.\n')
    f.write(f'- DINO@1: {top1:.4f}\n')
    f.write(f'- entropy: {entropy:.3f}\n')
    f.write(f'- dominant: {dominant * 100:.1f}%\n')
    f.write(f'- checkpoint: {row["dirname"]}\n')

subprocess.run(['git', 'add', 'PROGRESS.md'], check=True)
subprocess.run(['git', 'commit', '-m', f'Update Track A S24 r={RUN_RANK} results'], check=True)
subprocess.run(['git', 'push', 'origin', 'main'], check=True)
print('GitHub push complete.')
